# 薪资清洗并删除字段

将上一阶段清洗后的招聘数据继续处理：把薪资统一换算为年薪（单位：万元/年），并删除不需要的字段。

In [1]:
import re
from pathlib import Path

import pandas as pd


# 手动修改这里的路径，然后直接运行全部单元格。
# 路径可以改成任意 CSV 文件的绝对路径。
NATIONAL_SOURCE = Path(r'D:\桌面\实训项目\数据清洗\合并并筛选\清洗后的数据\全国范围_数据开发_清洗.csv')
CUSTOM_SOURCE = Path(r'D:\桌面\实训项目\数据清洗\合并并筛选\清洗后的数据\自定义范围_数据开发_清洗.csv')

# 输出文件固定为你要求的这两个位置。
OUTPUT_ROOT = Path(r'D:\桌面\实训项目\数据清洗\薪资清洗并删除\清洗后的数据')
NATIONAL_OUTPUT = OUTPUT_ROOT / '全国范围_数据开发_清洗.csv'
CUSTOM_OUTPUT = OUTPUT_ROOT / '自定义范围_数据开发_清洗.csv'

DROP_COLUMNS = [
    '薪资发放次数',
    '工作城市', '行政区', '商圈/街道',
    '详细地址', '经度', '纬度',
    '福利标签', '福利明细', '工作时间', '报告项/保障项',
]

DEFAULT_MONTHS = 12
WORK_DAYS_PER_MONTH = 25
HOURS_PER_DAY = 8

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('全国范围源文件:', NATIONAL_SOURCE)
print('自定义范围源文件:', CUSTOM_SOURCE)


全国范围源文件: D:\桌面\实训项目\数据清洗\合并并筛选\清洗后的数据\全国范围_数据开发_清洗.csv
自定义范围源文件: D:\桌面\实训项目\数据清洗\合并并筛选\清洗后的数据\自定义范围_数据开发_清洗.csv


In [2]:
def format_wan(value: float) -> str:
    """格式化万元数值，去掉多余的 0。"""
    text = f'{float(value):.2f}'
    return text.rstrip('0').rstrip('.')


def format_salary_range(min_wan: float, max_wan: float) -> str:
    if abs(min_wan - max_wan) < 0.005:
        return f'{format_wan(min_wan)}万'
    return f'{format_wan(min_wan)}-{format_wan(max_wan)}万'


def parse_salary_to_annual_wan(text):
    """把薪资文本解析成年薪，返回：下限、上限、换算说明。单位为万元/年。"""
    raw = '' if pd.isna(text) else str(text).strip()
    if raw == '' or '面议' in raw:
        return None, None, '面议/空值，后续用均值回填'

    normalized = (
        raw.replace('－', '-')
        .replace('—', '-')
        .replace('–', '-')
        .replace('~', '-')
        .replace('～', '-')
        .replace('至', '-')
    )

    month_match = re.search(r'(\d+(?:\.\d+)?)\s*薪', normalized)
    months = int(float(month_match.group(1))) if month_match else DEFAULT_MONTHS
    salary_body = re.sub(r'[·・\s]*\d+(?:\.\d+)?\s*薪.*$', '', normalized)

    numbers = [float(item) for item in re.findall(r'\d+(?:\.\d+)?', salary_body)]
    if not numbers:
        return None, None, '无法解析，后续用均值回填'

    low = numbers[0]
    high = numbers[1] if len(numbers) >= 2 else numbers[0]
    if low > high:
        low, high = high, low

    if '元/天' in salary_body or '/天' in salary_body:
        factor = WORK_DAYS_PER_MONTH * 12 / 10000
        return low * factor, high * factor, f'日薪*{WORK_DAYS_PER_MONTH}天*12月/10000'

    if '元/时' in salary_body or '/时' in salary_body or '元/小时' in salary_body or '/小时' in salary_body:
        factor = HOURS_PER_DAY * WORK_DAYS_PER_MONTH * 12 / 10000
        return low * factor, high * factor, f'时薪*{HOURS_PER_DAY}小时*{WORK_DAYS_PER_MONTH}天*12月/10000'

    if re.search(r'[kK]', salary_body):
        factor = 1000 * months / 10000
        return low * factor, high * factor, f'月薪K*{months}薪/10'

    if '万' in salary_body:
        return low * months, high * months, f'月薪万*{months}薪'

    if '千' in salary_body:
        return low * 0.1 * months, high * 0.1 * months, f'月薪千*0.1*{months}薪'

    if '元' in salary_body:
        factor = months / 10000
        return low * factor, high * factor, f'月薪元*{months}薪/10000'

    if max(low, high) >= 1000:
        factor = months / 10000
        return low * factor, high * factor, f'按月薪元*{months}薪/10000'

    return low * months, high * months, f'按月薪万*{months}薪'


def add_annual_salary_columns(df: pd.DataFrame) -> tuple:
    result = df.copy()
    original_salary = result['薪资'].copy()

    parsed = original_salary.apply(parse_salary_to_annual_wan)
    result['年薪下限(万)'] = parsed.apply(lambda item: item[0])
    result['年薪上限(万)'] = parsed.apply(lambda item: item[1])
    result['薪资换算说明'] = parsed.apply(lambda item: item[2])
    result['年薪平均(万)'] = result[['年薪下限(万)', '年薪上限(万)']].mean(axis=1)

    annual_average = result['年薪平均(万)'].dropna().mean()
    fill_mask = result['年薪平均(万)'].isna()
    if fill_mask.any():
        result.loc[fill_mask, '年薪下限(万)'] = annual_average
        result.loc[fill_mask, '年薪上限(万)'] = annual_average
        result.loc[fill_mask, '年薪平均(万)'] = annual_average
        result.loc[fill_mask, '薪资换算说明'] = '面议/无法解析，使用非面议年薪平均数回填'

    for col in ['年薪下限(万)', '年薪上限(万)', '年薪平均(万)']:
        result[col] = result[col].round(2)

    result['原始薪资'] = original_salary
    result['薪资'] = result.apply(lambda row: format_salary_range(row['年薪下限(万)'], row['年薪上限(万)']), axis=1)

    insert_after = result.columns.get_loc('薪资') + 1
    new_columns = ['原始薪资', '年薪下限(万)', '年薪上限(万)', '年薪平均(万)', '薪资换算说明']
    ordered_columns = [col for col in result.columns if col not in new_columns]
    for offset, col in enumerate(new_columns):
        ordered_columns.insert(insert_after + offset, col)
    result = result[ordered_columns]

    stats = {
        'rows': len(result),
        'filled_by_average': int(fill_mask.sum()),
        'annual_average_used': round(float(annual_average), 2),
    }
    return result, stats


In [3]:
def clean_one_file(source_path: Path, output_path: Path) -> tuple:
    df = pd.read_csv(source_path, encoding='utf-8-sig', dtype=str).fillna('')
    cleaned, stats = add_annual_salary_columns(df)

    existing_drop_columns = [col for col in DROP_COLUMNS if col in cleaned.columns]
    cleaned = cleaned.drop(columns=existing_drop_columns)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    cleaned.to_csv(output_path, index=False, encoding='utf-8-sig')

    stats.update({
        'source': str(source_path),
        'output': str(output_path),
        'columns': len(cleaned.columns),
        'dropped_columns': existing_drop_columns,
    })
    return cleaned, stats


custom_df, custom_stats = clean_one_file(CUSTOM_SOURCE, CUSTOM_OUTPUT)
national_df, national_stats = clean_one_file(NATIONAL_SOURCE, NATIONAL_OUTPUT)

for name, stats in [('自定义范围数据', custom_stats), ('全国范围数据', national_stats)]:
    print('\n' + name)
    print('源文件:', stats['source'])
    print('输出文件:', stats['output'])
    print('行数:', stats['rows'])
    print('列数:', stats['columns'])
    print('面议/无法解析回填行数:', stats['filled_by_average'])
    print('回填使用年薪均值:', stats['annual_average_used'], '万')
    print('删除字段:', '、'.join(stats['dropped_columns']))

custom_df.head()



自定义范围数据
源文件: D:\桌面\实训项目\数据清洗\合并并筛选\清洗后的数据\自定义范围_数据开发_清洗.csv
输出文件: D:\桌面\实训项目\数据清洗\薪资清洗并删除\清洗后的数据\自定义范围_数据开发_清洗.csv
行数: 1831
列数: 32
面议/无法解析回填行数: 71
回填使用年薪均值: 15.52 万
删除字段: 薪资发放次数、工作城市、行政区、商圈/街道、详细地址、经度、纬度、福利标签、福利明细、工作时间、报告项/保障项

全国范围数据
源文件: D:\桌面\实训项目\数据清洗\合并并筛选\清洗后的数据\全国范围_数据开发_清洗.csv
输出文件: D:\桌面\实训项目\数据清洗\薪资清洗并删除\清洗后的数据\全国范围_数据开发_清洗.csv
行数: 1000
列数: 32
面议/无法解析回填行数: 54
回填使用年薪均值: 16.18 万
删除字段: 薪资发放次数、工作城市、行政区、商圈/街道、详细地址、经度、纬度、福利标签、福利明细、工作时间、报告项/保障项


,职位名称,薪资,原始薪资,年薪下限(万),年薪上限(万),年薪平均(万),薪资换算说明,薪资类型,工作地点展示,经验要求,...,是否新职位,招聘人数,HR状态,HR活跃标签,职位标签汇总,技能标签,职位描述,职位亮点,职位摘要,认证/守护信息
0,银行开发工程师 / 数据分析师（金融方向）,12-18万,1-1.5万,12.00,18.00,15.00,月薪万*12薪,1,上海 浦东 陆家嘴,经验不限,...,0,10,,,基金数据分析 | SQL | C++ | MySQL | 本科 | 银行,基金数据分析 | SQL | C++ | MySQL | 本科 | 银行,岗位职责<div> \n1. 负责银行内部业务报表的需求沟通、设计、开发与测试工作。 \n2...,,,实名核验 ・ 资质核验
1,数据开发（微软云经验）,26.4-30万,2.2-2.5万,26.40,30.00,28.20,月薪万*12薪,1,上海 静安 石门二路,5-10年,...,0,0,1分钟前回复,1分钟内回复 | 今日回复5次,Azure | 本科 | 5-10年,Azure | 本科 | 5-10年,良好的沟通和演示技巧，能够清晰沟通并向他人解释技术解决方案。\n一个快速学习者，出色的队友。...,,,
2,数据开发,9.6-15.6万,8000-13000元,9.60,15.60,12.60,月薪元*12薪/10000,1,上海 崇明 南京东路,经验不限,...,0,0,57分钟前回复,57分钟前回复 | 今日回复50+次,专人带教 | 五险一金 | 大牛带队,专人带教 | 五险一金,岗位职责\n通过线上聊天渠道解答用户常规疑问，做好咨询内容登记。\n整理用户高频问题汇总上交...,,,
3,服务零售-数据开发工程师,15.52万,面议,15.52,15.52,15.52,面议/无法解析，使用非面议年薪平均数回填,1,上海 杨浦 大桥,3-5年,...,0,1,刚刚活跃,,本科 | 3-5年 | 快速提升 | 互联网100强 | 最佳雇主,本科 | 3-5年 | 快速提升,基础研发平台是美团的核心技术平台，立足于“零售+科技”的战略定位，通过打造人工智能、大数据、...,,,
4,Java/ Web/ C开发/ 测试/算法/数据开发,22.4-44.8万,1.4-2.8万·16薪,22.40,44.80,33.60,月薪万*16薪,1,上海 浦东 金桥,经验不限,...,0,2,6小时内回复可能性大,今日回复11次,JavaScript | Python | C++ | Spring | Mybatis |...,JavaScript | Python | C++ | Spring | Mybatis |...,岗位要求】\n1、 本科以上毕业；\n2、 具备Java/python/C++/C任意一种开...,,,
